# Scientific Report: Unified Forecasting and VaR Backtesting Statistics

**Objective.** This notebook reproduces the statistical analysis used to evaluate volatility forecasts and risk-control quality from `output/merged_all_predictions.csv`.

**Scope.** The analysis reports forecasting accuracy (`MSE`, `MAE`, `QLIKE`) and risk calibration (`Violation Rate`, `Kupiec`, `Christoffersen Independence`, `Pass Rate`) across datasets, horizons, model families, and model tiers.

**Output.** All generated tables are saved to `output/stats_analysis` for paper-ready inspection and downstream reporting.

## 1. Evaluation Methodology

Let $y_t$ be the realized volatility target, $\hat{y}_t$ the predicted volatility, $r_t$ the realized return, and $n$ the number of valid evaluation observations.

Forecasting accuracy is measured as:

$$MSE = \frac{1}{n}\sum_{t=1}^{n}(y_t - \hat{y}_t)^2$$

$$MAE = \frac{1}{n}\sum_{t=1}^{n}|y_t - \hat{y}_t|$$

$$QLIKE = \frac{1}{n}\sum_{t=1}^{n}\left(\frac{y_t}{\hat{y}_t} - \log\left(\frac{y_t}{\hat{y}_t}\right) - 1\right)$$

Risk quality is evaluated at the 5% left tail using Student-t VaR:

$$\widehat{VaR}^{\alpha}_{t} = \mu + \hat{\sigma}_{t}q_{\alpha}$$

$$I_t = \mathbf{1}(r_t < \widehat{VaR}^{\alpha}_{t})$$

A dataset-horizon case passes the VaR backtest when both `kupiec_p > 0.05` and `lr_ind_p > 0.05`.

## 2. Reproducibility Setup

This cell resolves the project root, imports the reusable `stats_analysis` package, and defines the report configuration. The notebook is intentionally thin: all statistical logic lives in the package modules for maintainability and testability.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "output" / "merged_all_predictions.csv").exists():
    PROJECT_ROOT = Path("D:/UIT_LEARNING_MATERIAL/07.Research/code")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from stats_analysis import StatsAnalysisConfig, StatsAnalysisPipeline

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.6f}".format)

risk_configs = {
    "var_5pct": StatsAnalysisConfig(project_root=PROJECT_ROOT, alpha=0.05),
    "var_1pct": StatsAnalysisConfig(
        project_root=PROJECT_ROOT,
        alpha=0.01,
        output_dir=PROJECT_ROOT / "output" / "stats_analysis" / "var_1pct",
    ),
}
config = risk_configs["var_5pct"]

config_table = pd.DataFrame([
    {"parameter": "project_root", "value": str(config.project_root)},
    {"parameter": "input_csv", "value": str(config.resolved_input_csv())},
    {"parameter": "output_dir", "value": str(config.resolved_output_dir())},
    {"parameter": "alpha", "value": config.alpha},
    {"parameter": "var_1pct_output_dir", "value": str(risk_configs["var_1pct"].resolved_output_dir())},
    {"parameter": "pvalue_threshold", "value": config.pvalue_threshold},
    {"parameter": "student_t_min_nu", "value": config.student_t_min_nu},
    {"parameter": "epsilon", "value": config.epsilon},
])

config_table

## 3. Execute Analysis Pipeline

The pipeline reads the merged prediction file, validates required columns, reports missing values directly, computes all metrics, and writes the result tables.

In [ ]:
results = {
    name: StatsAnalysisPipeline(risk_config).run(save=True)
    for name, risk_config in risk_configs.items()
}

result = results["var_5pct"]
result_var_1pct = results["var_1pct"]

summary = result.summary.copy()
detailed = result.detailed.copy()
aggregate = result.aggregate.copy()
pass_cases = result.pass_cases.copy()

summary

## 4. Data Audit

This section checks whether the analysis covers the expected benchmark design: nine datasets, five forecast horizons, and all models found in the merged prediction file.

In [ ]:
coverage = pd.DataFrame([
    {"item": "datasets", "count": detailed["dataset"].nunique(), "values": ", ".join(sorted(detailed["dataset"].astype(str).unique()))},
    {"item": "horizons", "count": detailed["horizon"].nunique(), "values": ", ".join(map(str, sorted(detailed["horizon"].unique())))},
    {"item": "models", "count": detailed["model"].nunique(), "values": ", ".join(sorted(detailed["model"].astype(str).unique()))},
    {"item": "model groups", "count": len(aggregate), "values": "branch/tier/model combinations"},
])

coverage

In [ ]:
audit_columns = [
    "input_rows",
    "missing_return",
    "missing_true_volatility",
    "forecast_invalid_rows",
    "risk_invalid_rows",
]

summary[audit_columns].T.rename(columns={0: "value"})

## 5. Aggregate Results by Model

Lower values indicate better forecasting accuracy for `MSE`, `MAE`, and `QLIKE`. Higher `pass_rate` indicates better statistical acceptability in VaR backtesting.

In [ ]:
aggregate_report = aggregate[[
    "branch",
    "tier",
    "model",
    "cases",
    "valid_risk_cases",
    "pass_rate",
    "qlike",
    "mse",
    "mae",
    "violation_rate",
    "kupiec_p",
    "lr_ind_p",
]].copy()

aggregate_report.sort_values(["pass_rate", "qlike"], ascending=[False, True]).reset_index(drop=True)

## 6. Ranking Summary

The following compact tables identify the strongest models under the two evaluation objectives: forecasting accuracy and VaR backtesting reliability.

In [ ]:
rank_columns = ["branch", "tier", "model", "pass_rate", "qlike", "mse", "mae", "violation_rate"]

best_by_pass_rate = aggregate.sort_values(["pass_rate", "qlike"], ascending=[False, True])[rank_columns].head(10)
best_by_qlike = aggregate.sort_values(["qlike", "pass_rate"], ascending=[True, False])[rank_columns].head(10)

display(best_by_pass_rate.style.set_caption("Top models by VaR backtest pass rate"))
display(best_by_qlike.style.set_caption("Top models by QLIKE"))

## 7. Forecasting-Risk Trade-off

This plot visualizes whether lower forecasting error aligns with higher risk backtesting pass rate. A mismatch supports the paper's central claim that forecasting accuracy alone does not guarantee risk-control quality.

In [ ]:
plot_df = aggregate.dropna(subset=["qlike", "pass_rate"]).copy()
plot_df["label"] = plot_df.apply(
    lambda row: row["model"] if row["tier"] == "" else f"{row['model']} ({row['tier']})",
    axis=1,
)

if plt is None:
    display(plot_df[["label", "qlike", "pass_rate"]].sort_values("pass_rate", ascending=False))
else:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(plot_df["qlike"], plot_df["pass_rate"], s=80, alpha=0.75)
    for _, row in plot_df.iterrows():
        ax.annotate(row["label"], (row["qlike"], row["pass_rate"]), xytext=(5, 4), textcoords="offset points", fontsize=8)
    ax.set_xlabel("QLIKE (lower is better)")
    ax.set_ylabel("Pass rate (higher is better)")
    ax.set_title("Forecasting Accuracy vs VaR Backtesting Reliability")
    ax.grid(True, alpha=0.25)
    plt.show()

## 8. Detailed Dataset-Horizon Results

Each row below corresponds to one `branch/tier/model/dataset/horizon` case. These rows are the basis for aggregate pass-rate computation.

In [ ]:
detailed_report = detailed[[
    "branch",
    "tier",
    "model",
    "dataset",
    "horizon",
    "n_forecast",
    "n_risk",
    "mse",
    "mae",
    "qlike",
    "violation_rate",
    "kupiec_p",
    "lr_ind_p",
    "backtest_pass",
]].copy()

detailed_report.head(30)

## 9. Backtest Pass Cases

This table isolates the pass/fail decision for Kupiec coverage, Christoffersen independence, and the combined backtest criterion.

In [ ]:
pass_cases.sort_values(["backtest_pass", "model", "dataset", "horizon"], ascending=[False, True, True, True]).head(30)

## 10. Output Manifest

The pipeline writes four CSV files. These files are the canonical artifacts for tables, appendix inspection, and paper revision.

In [ ]:
output_dir = config.resolved_output_dir()
manifest = pd.DataFrame([
    {"artifact": "stats_by_dataset_horizon", "path": str(output_dir / config.detailed_filename)},
    {"artifact": "stats_by_model", "path": str(output_dir / config.aggregate_filename)},
    {"artifact": "backtest_pass_cases", "path": str(output_dir / config.pass_cases_filename)},
    {"artifact": "stats_run_summary", "path": str(output_dir / config.summary_filename)},
])

manifest["exists"] = manifest["path"].map(lambda p: Path(p).exists())
manifest["size_bytes"] = manifest["path"].map(lambda p: Path(p).stat().st_size if Path(p).exists() else 0)
manifest

## 11. Interpretation Notes

- `QLIKE`, `MSE`, and `MAE` answer the point-forecasting question: how close predicted volatility is to realized volatility.
- `Violation Rate`, `Kupiec`, and `Christoffersen Independence` answer the risk-control question: whether VaR violations are calibrated and temporally acceptable.
- `Pass Rate` summarizes how often a model passes both VaR backtests across dataset-horizon cases.
- Rows with missing `log_return` are excluded only from risk metrics, not from forecast metrics. The current merged CSV has complete `log_return`; remaining forecast invalid rows come from missing `true_volatility`.